# 01 - 设置与基础

本 notebook 准备了本教程其余部分使用的本地文档语料库。

我们将:
- 检查项目配置
- 下载 LangChain 源页面到项目本地
- 加载本地文本文件作为文档
- 将它们分割成块
- 比较不同的分块策略

**前置条件:** 无

**输出:**
- `data/source_docs/langchain/` 下的本地源文件
- 内存中的文档块,供 notebook 02 使用

## 1. 环境与配置

在下载和分割文档之前检查活动配置。

In [11]:
import sys
sys.path.append('../..')

from shared.config import SECTION_WIDTH, get_project_info, verify_api_key
from shared.utils import print_section_header

print_section_header('环境设置')

api_key_ok = verify_api_key()

print('\n项目配置:')
print('-' * SECTION_WIDTH)
info = get_project_info()
for key, value in info.items():
    print(f'{key:.<30} {value}')

print('\n环境设置完成')


环境设置

OK deepseek API key: LOADED
  Preview: sk-0501...20ae
  Base URL: https://api.deepseek.com

项目配置:
--------------------------------------------------------------------------------
environment................... dev
debug_mode.................... False
log_level..................... INFO
project_root.................. d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..
vector_store_dir.............. d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\vector_stores
cache_dir..................... d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\cache
llm_provider.................. deepseek
embedding_provider............ huggingface
llm_api_key_loaded............ True
llm_base_url.................. https://api.deepseek.com
openai_api_key_loaded......... False
deepseek_api_key_loaded....... True
huggingface_api_key_loaded.... False
tavily_api_key_loaded......... False
langsmith_api_key_loaded...... False
default_model..........

## 2. 下载文档到项目

首先下载源页面,以便后续的 notebook 从本地文件工作,而不依赖于实时页面获取。

In [12]:
from shared.config import DEFAULT_LANGCHAIN_URLS
from shared.loaders import download_langchain_docs_to_local

print_section_header('下载源文档')

local_paths = download_langchain_docs_to_local(
    urls=DEFAULT_LANGCHAIN_URLS,
    verbose=True,
)

print('\n已下载的文件:')
for path in local_paths:
    print(f'  - {path}')


下载源文档

  - Target: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain
  - https://python.langchain.com/docs/use_cases/question_answering/
  - https://python.langchain.com/docs/modules/data_connection/retrievers/
  - https://python.langchain.com/docs/modules/model_io/llms/
  - https://python.langchain.com/docs/use_cases/chatbots/
OK Downloaded 4 local documents

已下载的文件:
  - d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain\01_question_answering.txt
  - d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain\02_retrievers.txt
  - d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain\03_llms.txt
  - d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain\04_chatbots.txt


## 3. 加载本地文档

将下载的本地文本文件加载为 LangChain `Document` 对象。

In [13]:
from shared.loaders import load_local_text_docs

print_section_header('加载本地文档')

docs = load_local_text_docs(
    local_paths=local_paths,
    original_urls=DEFAULT_LANGCHAIN_URLS,
    verbose=True,
)

sample_doc = docs[0]
print('\n示例文档元数据:')
print('-' * SECTION_WIDTH)
print(f"来源: {sample_doc.metadata.get('source', 'N/A')}")
print(f"本地路径: {sample_doc.metadata.get('local_path', 'N/A')}")
print(f"来源类型: {sample_doc.metadata.get('source_type', 'N/A')}")
print(f"处理日期: {sample_doc.metadata.get('process_date', 'N/A')}")
print(f"\n预览:\n{sample_doc.page_content[:300]}...")


加载本地文档

OK Loaded 4 local documents
OK Added custom metadata to all documents

示例文档元数据:
--------------------------------------------------------------------------------
来源: https://python.langchain.com/docs/use_cases/question_answering/
本地路径: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain\01_question_answering.txt
来源类型: local_text_download
处理日期: 2026-05-09

预览:
Build a RAG agent with LangChain - Docs by LangChain
Skip to main content
Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain.
Buy tickets >
Docs by LangChain
home page
Open source
Search...
⌘
K
Ask AI
GitHub
Try LangSmith
Try LangSmith
Search...
Navigation
LangChain
Build a ...


## 4. 分割文档

将本地文档分割成用于检索的块。

In [14]:
from shared.config import DEFAULT_CHUNK_OVERLAP, DEFAULT_CHUNK_SIZE
from shared.loaders import split_documents

print_section_header('分割文档')

chunks = split_documents(
    docs,
    chunk_size=DEFAULT_CHUNK_SIZE,
    chunk_overlap=DEFAULT_CHUNK_OVERLAP,
    verbose=True,
)

print(f'\n创建了 {len(chunks)} 个块')
print(f'块大小={DEFAULT_CHUNK_SIZE}, 重叠={DEFAULT_CHUNK_OVERLAP}')


分割文档

Splitting documents...
  - Chunk size: 1000
  - Chunk overlap: 200
✓ Created 149 chunks

  Sample chunk:
    - Length: 981 chars
    - Source: https://python.langchain.com/docs/use_cases/question_answering/
    - Preview: Build a RAG agent with LangChain - Docs by LangChain
Skip to main content
Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain....

创建了 149 个块
块大小=1000, 重叠=200


## 5. 比较分块策略

将默认分块策略与较小的块配置进行比较。

In [15]:
from shared.loaders import compare_splitting_strategies

print_section_header('分块策略比较')

strategies = [
    (1000, 200),
    (500, 100),
]

results = compare_splitting_strategies(docs, strategies, verbose=True)

print('\n建议:')
print('  - 使用 1000/200 获取更广泛的上下文')
print('  - 使用 500/100 进行更细粒度的检索')
print('  - 本教程继续使用 1000/200')


分块策略比较


=== Splitting Strategy Comparison ===

Strategy        Chunk Size      Overlap         Chunks    
------------------------------------------------------------
1000/200        1000            200             149       
500/100         500             100             302       

💡 Larger chunks = more context, fewer chunks
💡 Smaller chunks = more precise, more chunks

建议:
  - 使用 1000/200 获取更广泛的上下文
  - 使用 500/100 进行更细粒度的检索
  - 本教程继续使用 1000/200


## 总结

在本 notebook 中,我们:
- 将源页面下载到项目目录
- 将这些本地文件加载为 LangChain 文档
- 将它们分割为检索块
- 比较了分块策略

下一步:
- 继续到 [02_embeddings_comparison.ipynb](02_embeddings_comparison.ipynb) 从相同的本地语料库构建嵌入和向量存储